In [12]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client ready.")

Client ready.


In [13]:
def call_openai(system_prompt, user_message, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content

## Step 2: Initial Prompts (Version 1 — Zero-Shot)

Three basic prompts, one test each. No optimization yet — we want to see raw output before any engineering.

In [14]:
# Task 1: Sentiment Analysis — Version 1
# Zero-shot, no format constraints, no output schema

sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

result = call_openai("You are a helpful assistant.", sentiment_prompt_v1)
print("Task 1 — Sentiment Analysis v1:")
print(result)

Task 1 — Sentiment Analysis v1:
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."


In [15]:
# Task 3: Data Extraction — Version 1
# Zero-shot, no schema, no format requirement

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

result = call_openai("You are a helpful assistant.", extraction_prompt_v1)
print("Task 3 — Data Extraction v1:")
print(result)

Task 3 — Data Extraction v1:
Here is the extracted information from the customer feedback:

- Item number: #12345
- Order date: March 15th
- Delivery: Fast
- Packaging condition: Damaged


In [20]:
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

## Step 3: Systematic Testing — 5 Runs

Run each v1 prompt 5 times to observe consistency. Are responses the same format? Same content? What varies?

In [19]:
def run_prompt_n_times(system_prompt, user_prompt, n=5):
    """Run the same prompt n times and return all results."""
    results = []
    for i in range(n):
        result = call_openai(system_prompt, user_prompt)
        results.append(result)
        print(f"Run {i+1}: {result}")
    
    unique = len(set(results))
    print(f"\n📊 Unique outputs: {unique}/{n}")
    print(f"📊 Consistency rate: {((n - unique + 1) / n * 100):.0f}%")
    return results

print("✅ run_prompt_n_times loaded")

✅ run_prompt_n_times loaded


In [17]:
print("="*60)
print("Task 1 — Sentiment Analysis v1 — 5 runs")
print("="*60)
sentiment_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    sentiment_prompt_v1,
    n=5
)

Task 1 — Sentiment Analysis v1 — 5 runs
Run 1: The customer message can be classified as "Positive feedback" or "Customer satisfaction."
Run 2: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 3: The customer message can be classified as **positive feedback** or **satisfaction**.
Run 4: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 5: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

📊 Unique outputs: 4/5
📊 Consistency rate: 40%


In [21]:
print("="*60)
print("Task 2 — Product Description v1 — 5 runs")
print("="*60)
product_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    product_prompt_v1,
    n=5
)

Task 2 — Product Description v1 — 5 runs
Run 1: **Product Description: Wireless Comfort Mouse - $29.99**

Say goodbye to tangled cords and hello to seamless navigation with our Wireless Comfort Mouse. Designed for productivity and comfort, this sleek and stylish mouse is the perfect companion for your laptop or desktop, whether at home, in the office, or on the go.

**Key Features:**

- **Ergonomic Design:** The Wireless Comfort Mouse features a contoured shape that fits snugly in your hand, reducing strain and enhancing comfort during extended use.

- **Reliable Wireless Connection:** Enjoy a clutter-free workspace with advanced wireless technology that provides a stable connection up to 33 feet away. No more interruptions — just smooth, accurate tracking.

- **High Precision Tracking:** Equipped with a high-resolution optical sensor, this mouse delivers precise cursor control on a variety of surfaces, making it ideal for both everyday tasks and detailed projects.

- **Long Battery Li

In [ ]:
print("="*60)
print("Task 3 — Data Extraction v1 — 5 runs")
print("="*60)
extraction_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    extraction_prompt_v1,
    n=5
)

Task 3 — Data Extraction v1 — 5 runs
Run 1: Here is the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 2: Here are the extracted details from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery Experience: Fast
- Packaging Condition: Damaged
Run 3: Here is the extracted information from the customer feedback:

- **Order Number:** 12345
- **Order Date:** March 15th
- **Delivery:** Fast
- **Packaging Condition:** Damaged
Run 4: Here’s the extracted information from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery: Fast
- Packaging Condition: Damaged
Run 5: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

📊 Unique outputs: 5/5
📊 Consistency rate: 20%


## Step 4: Systematic Testing — 10 Runs

Increasing to 10 runs to see if new failure patterns emerge.

In [22]:
print("="*60)
print("Task 1 — Sentiment Analysis v1 — 10 runs")
print("="*60)
sentiment_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    sentiment_prompt_v1,
    n=10
)

Task 1 — Sentiment Analysis v1 — 10 runs
Run 1: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 2: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 3: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 4: The customer message can be classified as "Positive Feedback" or "Satisfaction."
Run 5: This customer message can be classified as "Positive Feedback" or "Satisfaction."
Run 6: The customer message can be classified as **Positive Feedback** or **Customer Praise**.
Run 7: This customer message is positive feedback.
Run 8: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 9: The customer message can be classified as "Positive Feedback" or "Satisfaction" since the customer expresses their love for the product and confirms it meets their needs.
Run 10: This customer message can be classified as **positive feed

In [23]:
print("="*60)
print("Task 2 — Product Description v1 — 10 runs")
print("="*60)
product_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    product_prompt_v1,
    n=10
)

Task 2 — Product Description v1 — 10 runs
Run 1: **Product Description: Wireless Precision Mouse**

Elevate your everyday computing experience with our Wireless Precision Mouse, designed to deliver exceptional performance and comfort at an unbeatable price of just $29.99. Whether you're working from home, gaming, or browsing the web, this versatile mouse is the perfect companion for all your digital tasks.

**Key Features:**

- **Seamless Wireless Connectivity:** Enjoy the freedom of a clutter-free workspace with advanced 2.4GHz wireless technology. Simply plug in the USB receiver, and you're ready to go—no tangled wires, no hassle!

- **Ergonomic Design:** Crafted for comfort, the Wireless Precision Mouse features an ergonomic shape that fits snugly in your hand, reducing strain during extended use. The textured grip ensures optimal control, making it ideal for both office work and casual gaming.

- **High Precision Tracking:** With a DPI range adjustable from 800 to 2400, this mouse 

In [24]:
print("="*60)
print("Task 3 — Data Extraction v1 — 10 runs")
print("="*60)
extraction_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    extraction_prompt_v1,
    n=10
)

Task 3 — Data Extraction v1 — 10 runs
Run 1: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery:** Fast
- **Packaging Condition:** Damaged
Run 2: Here is the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 3: Here's the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 4: Customer Feedback Summary:
- Order Number: #12345
- Order Date: March 15th
- Delivery Speed: Fast
- Issue: Damaged packaging
Run 5: Here's the extracted information from the customer feedback:

- **Item Ordered**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Issue**: Damaged packaging
Run 6: Here’s the extracted information from the customer feedback:

- **Order Number**: #1